# 02 — Linear Mixed-Effects Modeling (LMM)

Linear Mixed-Effects Models (`statsmodels.MixedLM`) on the long-format survey
data: within-person growth across revision stages (**Pre → Draft → Edit**) and
the **Proficiency × Stage** interaction (the "Intermediate × Editing" effect).

**Sections**
1. Setup & data loading
2. Data preparation (predictor coding)
3. Baseline model: random intercepts per participant
4. Full model: Stage × Proficiency interaction & probing
5. Diagnostics: residuals, Q–Q, BLUPs, ICC
6. Publication-ready summary tables (CSV export)
7. Estimated marginal means

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats as sps
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.precision', 3)

CANDIDATES = ['../data/survey_long_format.csv', '../data/survey_raw_data.csv']
DATA_PATH = next(p for p in CANDIDATES if os.path.exists(p))
OUT = '../results'
os.makedirs(OUT, exist_ok=True)
print('Reading:', DATA_PATH)

## 1. Load & prepare data
- `Stage`: treatment-coded, reference = **Pre**
- `Proficiency`: reference = **Advanced**
- Random intercept per `ParticipantID` handles repeated measures.

In [ ]:
STAGES = ['Pre', 'Draft', 'Edit']

if 'long' in DATA_PATH:
    long_df = pd.read_csv(DATA_PATH)
else:
    wide = pd.read_csv(DATA_PATH)
    value_vars = [c for c in wide.columns if c.split('_')[0] in STAGES and c != 'WritingScore']
    long_df = wide.melt(id_vars=['ParticipantID', 'Proficiency'], value_vars=value_vars,
                        var_name='SS', value_name='Score')
    long_df['Stage'] = long_df['SS'].str.split('_').str[0]
    long_df['Strand'] = long_df['SS'].str.split('_').str[1]

long_df['Stage'] = pd.Categorical(long_df['Stage'], STAGES, ordered=True)
long_df['Proficiency'] = pd.Categorical(long_df['Proficiency'],
                                        ['Advanced', 'Intermediate', 'Upper-Intermediate'])
print(long_df.shape)
print(long_df.groupby('Proficiency', observed=True)['ParticipantID'].nunique())
long_df.head()

## 2. Baseline model — random intercept only
`Score ~ Stage +1 | ParticipantID)`)` on composite scores.

In [ ]:
comp = long_df[long_df.Strand == 'Composite'].copy()

m0 = smf.mixedlm('Score ~ Stage', comp, groups=comp['ParticipantID'])
res0 = m0.fit(reml=True, method='lbfgs')
print(res0.summary())

## 3. Full model — Stage × Proficiency interaction
Fixed effects for stage, proficiency, and their interaction; random intercept
per participant. Fitted with ML (reml=False) so fixed-effect comparisons are valid.

In [ ]:
m1 = smf.mixedlm('Score ~ Stage * Proficiency', comp, groups=comp['ParticipantID'])
res1 = m1.fit(reml=False, method='lbfgs')
print(res1.summary())

In [ ]:
# Strand-specific models (fixed Stage effect, random intercept)
for strand in ['Linguistic', 'Strategic', 'Critical']:
    d = long_df[long_df.Strand == strand]
    fit = smf.mixedlm('Score ~ Stage', d, groups=d['ParticipantID']).fit(reml=False, method='lbfgs')
    print(f'--- {strand}: Stage effects (vs Pre) ---')
    print(fit.params.filter(like='Stage'), '\n')

## 4. Interaction probing — Intermediate × Editing
Simple effects of **Edit vs Pre** (and Draft vs Pre) within each proficiency
group, with FDR-corrected p-values; plus interaction contrasts from the full model.

In [ ]:
rows = []
for prof in comp['Proficiency'].dropna().unique():
    d = comp[comp['Proficiency'] == prof]
    fit = smf.mixedlm('Score ~ Stage', d, groups=d['ParticipantID']).fit(reml=False, method='lbfgs')
    rows.append({'Proficiency': str(prof), 'n': d.ParticipantID.nunique(),
                 'Draft_vs_Pre_est': fit.params.get('Stage[T.Draft]', np.nan),
                 'Draft_vs_Pre_p': fit.pvalues.get('Stage[T.Draft]', np.nan),
                 'Edit_vs_Pre_est': fit.params.get('Stage[T.Edit]', np.nan),
                 'Edit_vs_Pre_p': fit.pvalues.get('Stage[T.Edit]', np.nan)})
simple = pd.DataFrame(rows).set_index('Proficiency')
simple['Edit_vs_Pre_p_FDR'] = multipletests(simple['Edit_vs_Pre_p'].fillna(1), method='fdr_bh')[1]
simple.round(4)

In [ ]:
# Interaction terms from the full model
ix = [i for i in res1.params.index if 'Proficiency' in i and 'Stage' in i]
pd.DataFrame({'estimate': res1.params[ix], 'SE': res1.bse[ix],
              'z': res1.tvalues[ix], 'p': res1.pvalues[ix]}).round(4)

## 5. Model diagnostics
Residuals vs fitted (heteroscedasticity), normal Q–Q, random-intercept BLUPs,
and the ICC from the null model.

In [ ]:
comp_fit = comp.dropna(subset=['Score']).copy()
comp_fit['fitted'] = res1.fittedvalues
comp_fit['resid'] = res1.resid

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
sns.scatterplot(data=comp_fit, x='fitted', y='resid', alpha=.5, ax=axes[0])
axes[0].axhline(0, color='red', ls='--')
axes[0].set(title='Residuals vs Fitted', xlabel='Fitted', ylabel='Residual')

(osm, osr), (slope, intercept, r) = sps.probplot(comp_fit['resid'], dist='norm')
axes[1].plot(osm, osr, '.', alpha=.5)
axes[1].plot(osm, slope * osm + intercept, 'r-')
axes[1].set(title='Normal Q-Q of residuals', xlabel='Theoretical quantiles', ylabel='Sample quantiles')

blup_vals = pd.concat({k: v.iloc[:, 0] for k, v in res0.random_effects.items()})
sns.histplot(blup_vals, bins=20, kde=True, ax=axes[2])
axes[2].set(title='Random intercepts (BLUPs)', xlabel='Deviation from fixed intercept')
plt.tight_layout(); plt.show()

In [ ]:
var_resid = res0.scale
var_group = float(res0.cov_re.iloc[0, 0])
icc = var_group / (var_group + var_resid)
print(f'Random-intercept variance: {var_group:.4f}')
print(f'Residual variance:         {var_resid:.4f}')
print(f'ICC = {icc:.3f}  ->  {icc*100:.1f}% of variance is between-participant')

## 6. Publication-ready summary tables
APA-style fixed-effects tables, exported to `../results/`.

In [ ]:
def apa_table(fit, model_name):
    t = pd.DataFrame({'b': fit.params, 'SE': fit.bse, 'z': fit.tvalues, 'p': fit.pvalues,
                      'CI2.5': fit.conf_int()[0], 'CI97.5': fit.conf_int()[1]})
    t.index.name = 'Term'
    t.insert(0, 'Model', model_name)
    return t.round(3)

summary_table = pd.concat([apa_table(res0, 'M0: random intercept'),
                           apa_table(res1, 'M1: Stage x Proficiency')])
print(summary_table)
summary_table.to_csv(os.path.join(OUT, 'lmm_fixed_effects_summary.csv'))
simple.round(4).to_csv(os.path.join(OUT, 'lmm_simple_effects_stage.csv'))
print('\nSaved tables to', os.path.abspath(OUT))

## 7. Estimated marginal means
Model-based means by Stage × Proficiency from the full model.

In [ ]:
grid = pd.MultiIndex.from_product(
    [STAGES, ['Advanced', 'Intermediate', 'Upper-Intermediate']],
    names=['Stage', 'Proficiency']).to_frame(index=False)
grid['Stage'] = pd.Categorical(grid['Stage'], STAGES, ordered=True)
grid['Proficiency'] = pd.Categorical(grid['Proficiency'],
                                     ['Advanced', 'Intermediate', 'Upper-Intermediate'])
grid['emmean'] = res1.predict(grid)

plt.figure(figsize=(8, 4.5))
sns.lineplot(data=grid, x='Stage', y='emmean', hue='Proficiency', marker='o')
plt.ylabel('Estimated marginal mean (Composite)')
plt.title('Model-based means: Stage x Proficiency')
plt.tight_layout(); plt.show()
grid.round(3)

## Summary
- **M0** quantifies within-person stage growth and the ICC (clustering).
- **M1** tests whether stage effects differ by proficiency — the
  *Intermediate × Edit* interaction is read directly from the fixed-effects table.
- Simple-effects probing (Section 4) shows within which groups the Pre → Edit
  change is significant (FDR-corrected).
- Diagnostics (residual plots, Q–Q, BLUPs, ICC) support model adequacy.
- Tables exported to `../results/lmm_fixed_effects_summary.csv` and
  `../results/lmm_simple_effects_stage.csv`.